In [1]:
import json
import random
from pathlib import Path


### LAMBADA

In [2]:
from datasets import load_dataset

ds = load_dataset(
    "json",
    data_files="https://huggingface.co/datasets/EleutherAI/lambada_openai/resolve/main/data/lambada_test_en.jsonl"
)

print(ds)


/home/jl3499/conda/LLM1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 5153
    })
})


In [3]:


N = 1000
SEED = 42
MIN_WORDS = 50
QUOTE_CHARS = '"\'"\u201c\u201d\u2018\u2019'

def strip_trailing_punct(text):
    return text.rstrip('.,;:?!\'")\u201d\u201c\u2019\u2018')

def is_valid_example(text):
    # if any(c in text for c in QUOTE_CHARS):
    #     return False
    if len(text.split()) < MIN_WORDS:
        return False
    return True

random.seed(SEED)
train_data = ds["train"]
indices = list(range(len(train_data)))
random.shuffle(indices)
samples = []
for i in indices:
    if len(samples) >= N:
        break
    ex = train_data[int(i)]
    text = ex["text"]
    # print(len(text.split()))
    if not is_valid_example(text):
        continue
    cleaned = strip_trailing_punct(text.strip())
    samples.append({"text": cleaned})
assert len(samples) == N, f"Could not find {N} examples without quotes and with >= {MIN_WORDS} words"



In [4]:
# # out_path = Path("lambada_prompts.json")
# out_path = Path("lambada_prompts_1000.json")
# out_path.parent.mkdir(parents=True, exist_ok=True)
# with open(out_path, "w", encoding="utf-8") as f:
#     json.dump(samples, f, indent=2, ensure_ascii=False)
# print(f"Saved {len(samples)} examples to {out_path}")

### Translation: WMT/IWSLT


In [20]:
import json
import re
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

N = 1000

url = "https://huggingface.co/datasets/IWSLT/iwslt2017/resolve/main/data/2017-01-trnted/texts/de/en/de-en.zip"
zip_path = Path("de-en.zip")
extract_dir = Path("iwslt2017_de_en")
out_path = Path("IWSLT2017DE_EN.json")

if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

de_file = next(extract_dir.rglob("train.tags.de-en.de"))
en_file = next(extract_dir.rglob("train.tags.de-en.en"))

tag_re = re.compile(r"<[^>]+>")
trail_punct_re = re.compile(r"[^\w]+$", re.UNICODE)

def clean_lines(path):
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s or (s.startswith("<") and s.endswith(">")):
                continue
            s = tag_re.sub("", s).strip()
            if s:
                lines.append(s)
    return lines

german = clean_lines(de_file)
english = clean_lines(en_file)

data = []
for de, en in zip(german, english):
    en = trail_punct_re.sub("", en).strip()
    if de and en and len(en.split()) >= 20:
        # data.append({
        #     "German": de,
        #     "English": en,
        # })
        data.append({"text": f"German: {de} English: {en}"})
    if len(data) == N:
        break



In [21]:
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved {len(data)} pairs to {out_path}")
print(data[0])

Saved 1000 pairs to IWSLT2017DE_EN.json
{'text': 'German: Ich bin wirklich begeistert von dieser Konferenz, und ich danke Ihnen allen für die vielen netten Kommentare zu meiner Rede vorgestern Abend. English: I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night'}
